In [3]:
!uv pip install tokenizers

Using Python 3.10.20 environment at: D:\Ai engineering course\.venv
Checked 1 package in 55ms


In [4]:
!uv pip install transformers

Using Python 3.10.20 environment at: D:\Ai engineering course\.venv
Checked 1 package in 15ms


In [5]:


# from collections import Counter



# def merge(vocab, winning_pair):
#     new_vocab = {} # This will be our new dictionary with the glued words
    
#     # Go through every chopped-up word in our current dictionary
#     for word_tuple, freq in vocab.items():
#         new_word = []
#         i = 0
        
#         # Look at the letters one by one
#         while i < len(word_tuple):
#             # Check if the current letter AND the next letter match our winning pair
#             if i < len(word_tuple) - 1 and word_tuple[i] == winning_pair[0] and word_tuple[i+1] == winning_pair[1]:
#                 # GLUE THEM TOGETHER! (e.g., 'e' + 'r' becomes 'er')
#                 new_word.append(winning_pair[0] + winning_pair[1])
#                 i += 2 # We skip a step because we just combined two letters into one!
#             else:
#                 # No match, just keep the single letter as it is
#                 new_word.append(word_tuple[i])
#                 i += 1
                
#         # Save the newly glued word back into our dictionary
#         new_vocab[tuple(new_word)] = freq
        
#     return new_vocab

# corpus = "low lower lowest slow slower newest widest new news"  # built here

# # every word starts as characters + an end marker "_"
# vocab = {tuple(list(w) + ["_"]): c for w, c in Counter(corpus.split()).items()}

# for step in range(6):
#     pairs = Counter()
#     for word, freq in vocab.items():
#         for a, b in zip(word[:-1], word[1:]): pairs[(a, b)] += freq
#     best = pairs.most_common(1)[0][0]     # the most frequent pair
#     vocab = merge(vocab, best)               # glue it everywhere

In [6]:
from tokenizers import Tokenizer
from tokenizers.models import BPE
from tokenizers.trainers import BpeTrainer
from tokenizers.pre_tokenizers import Whitespace

reviews = [
    "the movie was amazing",
    "i really enjoyed this film",
    "the movie was terrible",
    "i loved the story",
    "this film was boring",
    "the movie was fantastic",
    "i did not like this movie",
    "the acting was excellent",
    "this was a great film",
    "i hated the ending",
    "the movie was disappointing",
    "i enjoyed every minute",
    "the film was awful",
    "the story was interesting",
    "i loved this movie",
    "the movie was not good",
    "the acting was really bad",
    "this film was wonderful",
    "i enjoyed the movie",
    "the movie was a complete waste of time"
]
tokenizer = Tokenizer(BPE(unk_token="[UNK]"))
tokenizer.pre_tokenizer = Whitespace()
trainer = BpeTrainer(vocab_size=40, special_tokens=["[UNK]"])
tokenizer.train_from_iterator(reviews, trainer)   # trains locally in a blink

In [7]:
tokenizer.encode("the movie was amazing").tokens

['the', 'movie', 'was', 'a', 'm', 'a', 'z', 'ing']

In [8]:
from transformers import pipeline
clf = pipeline("sentiment-analysis")          # downloads a trained model once
clf("movie was not so good")

D:\Career\Python\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
[transformers] No model was supplied, defaulted to distilbert/distilbert-base-uncased-finetuned-sst-2-english and revision 714eb0f.
Using a pipeline without specifying a model name and revision in production is not recommended.
Loading weights: 100%|██████████| 104/104 [00:00<00:00, 8099.67it/s]


[{'label': 'NEGATIVE', 'score': 0.9997442364692688}]

In [9]:
import torch
import torch.nn as nn
from tokenizers import Tokenizer
from tokenizers.models import BPE
from tokenizers.trainers import BpeTrainer
from tokenizers.pre_tokenizers import Whitespace

# ==========================================
# STEP 1: The Raw Data
# ==========================================
reviews = [
    "the movie was great", 
    "i hated this film", 
    "it was absolutely amazing", 
    "what a waste of time"
]
# 1 = Positive Review, 0 = Negative Review
labels = [1, 0, 1, 0] 


tokenizer = Tokenizer(BPE(unk_token="[UNK]"))
tokenizer.pre_tokenizer = Whitespace()

# Notice we added "[PAD]" as our very first special token so it gets ID 0
trainer = BpeTrainer(vocab_size=20, special_tokens=["[PAD]", "[UNK]"])
tokenizer.train_from_iterator(reviews, trainer)

# Translate words into raw IDs

    
    
print("Raw IDs:", raw_ids)





max_len = 25
padded_ids = []

for ids in raw_ids:
    # If the sentence is shorter than 5, calculate how many zeroes we need
    pad_amount = max_len - len(ids)
    # Add the zeroes to the end
    padded_ids.append(ids + [0] * pad_amount)

print(f"padded ids is: {padded_ids}")

# Put them inside PyTorch's high-speed Tensors


Y = torch.tensor(labels, dtype=torch.float32).reshape(-1, 1)

X = torch.tensor(padded_ids) 

# ==========================================
# STEP 4: The Brain (PyTorch Module)
# ==========================================
class SentimentNet(nn.Module):
    def __init__(self, vocab_size, embed_dim=25):
        super().__init__()
        # Translator: Turns ID numbers into rich mathematical meanings
        self.embedding = nn.Embedding(num_embeddings=vocab_size, embedding_dim=embed_dim)
        
        self.linear = nn.Linear(in_features=embed_dim, out_features=1)
        
        self.sigmoid = nn.Sigmoid() 

    def forward(self, x):
        embedded = self.embedding(x)             # Swap IDs for Embeddings
        averaged = embedded.mean(dim=1)          # Average the words to get the "Sentence Meaning"
        guess = self.sigmoid(self.linear(averaged)) # Make a final guess!
        return guess

# Build the factory!
model = SentimentNet(vocab_size=22)



# BCELoss is "Binary Cross Entropy". It is the specific math grader for 0 or 1 questions.
loss_fn = nn.BCELoss() 
optimizer = torch.optim.Adam(model.parameters(), lr=0.1)

print("\n--- Training Started ---")
for epoch in range(50):
    predictions = model(X)           # 1. The Guess
    loss = loss_fn(predictions, Y)   # 2. Check how wrong we are
    optimizer.zero_grad()            # 3. Clear the old blame scores
    loss.backward()                  # 4. Play the Blame Game (Backpropagation)
    optimizer.step()                 # 5. Adjust the Dials (Weights/Embeddings)
print("Training Complete! Final Error:", loss.item())




new_sentence = "the movie was amazing"


new_ids = tokenizer.encode(new_sentence).ids

new_padded = new_ids + [0] * (max_len - len(new_ids))

new_tensor = torch.tensor([new_padded])

# D. Predict!
with torch.no_grad(): # Freeze the brain, no learning allowed during a test!
    final_score = model(new_tensor)
    
    print(f"\nTest Sentence: '{new_sentence}'")
    print(f"AI Positivity Score (0 to 1): {final_score.item():.4f}")

NameError: name 'raw_ids' is not defined

In [ ]:


raw_ids = [tokenizer.encode(r).ids for r in reviews]
max_no_token = []
initial_count = 0

for i in reviews:
    
    if len(tokenizer.encode(i).ids)>initial_count:
        initial_count = len(tokenizer.encode(i).ids)



initial_count


In [ ]:

max_len = 25
padded_ids = []

for ids in raw_ids:
    # If the sentence is shorter than 5, calculate how many zeroes we need
    pad_amount = max_len - len(ids)
    # Add the zeroes to the end
    padded_ids.append(ids + [0] * pad_amount)

tokenizer.get_vocab_size()


In [ ]:
raw_ids

In [ ]:
import torch
import torch.nn as nn
from tokenizers import Tokenizer
from tokenizers.models import BPE
from tokenizers.trainers import BpeTrainer
from tokenizers.pre_tokenizers import Whitespace

# ==========================================
# STEP 1: The Raw Data
# ==========================================
reviews = [
    "the movie was great", 
    "i hated this film", 
    "it was absolutely amazing", 
    "what a waste of time"
    "faster than the car"
]
# 1 = Positive Review, 0 = Negative Review
labels = [1, 0, 1, 0] 


tokenizer = Tokenizer(BPE(unk_token="[UNK]"))
tokenizer.pre_tokenizer = Whitespace()

# Notice we added "[PAD]" as our very first special token so it gets ID 0
trainer = BpeTrainer(vocab_size=100, special_tokens=["[PAD]", "[UNK]"])
tokenizer.train_from_iterator(reviews, trainer)

# Translate words into raw IDs
raw_ids = [tokenizer.encode(r).ids for r in reviews]
print("Raw IDs:", raw_ids)

In [ ]:
new_ids.tokens

In [ ]:


raw_ids = [[28, 69, 26, 66, 34, 45, 34,22]]

max_len = 5
padded_ids = []

for ids in raw_ids:
    # If the sentence is shorter than 5, calculate how many zeroes we need
    pad_amount = max_len - len(ids)
    # Add the zeroes to the end
    padded_ids.append(ids + [0] * pad_amount)


In [ ]:
padded_ids

In [ ]:
[0]*(-5)